# Build a General-Purpose Neuron vLLM BYOC Container for SageMaker

Build a BYOC (Bring Your Own Container) for deploying **any** vllm-neuron supported model on SageMaker Neuron instances (inf2/trn1) using vLLM with NxD Inference.

The container is model-agnostic -- the same image deploys Llama, Qwen, Qwen-VL, Pixtral, etc. Model configuration is passed via environment variables at deploy time.

## What Gets Built

- **Base**: Official `pytorch-inference-vllm-neuronx` DLC (SDK 2.28, vLLM 0.13.0, vllm-neuron 0.4.1, NxDI 0.8.x)
- **Patches**: Sampler import fix, NxDI deepstack_vision_embeds forward fix, on-device sampling disabled
- **Serving**: FastAPI proxy (`entrypoint.py`) bridging SageMaker's `/ping` + `/invocations` to vLLM's native OpenAI API
- **Startup patches**: On-disk NxDI fixes for tie_word_embeddings and vision QKV split (applied before vLLM launches)

## Prerequisites

- **Docker available** (EC2, SageMaker notebook instance, or local machine)
- **AWS CLI configured** with ECR push permissions
- **~15 GB disk space** for the Docker build

## Why SDK 2.28?

SageMaker only offers inf2 and trn1 Neuron instances (no trn2). SDK 2.29 (NxD Inference 0.9) drops inf2/trn1 support, so SDK 2.28 is the latest compatible version.

## Step 1: Configuration

In [ ]:
import boto3
import os

sts = boto3.client('sts')
ACCOUNT_ID = sts.get_caller_identity()['Account']
REGION = boto3.Session().region_name

# Container configuration
ECR_REPO = "neuron-vllm-sagemaker"
ECR_TAG = "sdk2.28-v8"

IMAGE_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:{ECR_TAG}"

print(f"Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")
print(f"Target image: {IMAGE_URI}")

## Step 2: Create Build Directory

The container needs two files:

1. **Dockerfile** -- Based on the vLLM Neuron DLC, adds patches and the SageMaker proxy
2. **entrypoint.py** -- FastAPI server that bridges SageMaker's serving contract to vLLM, with on-disk NxDI patches

Both files are in the `sagemaker-byoc/` directory of this repository. The cells below write them to a build directory.

In [ ]:
BUILD_DIR = "/tmp/neuron-vllm-sagemaker-build"
os.makedirs(BUILD_DIR, exist_ok=True)

print(f"Build directory: {BUILD_DIR}")

## Step 3: Create Dockerfile

The Dockerfile does 4 things:

1. **Starts from the vLLM Neuron DLC** -- has vLLM 0.13.0, vllm-neuron 0.4.1, NxDI 0.8.x pre-installed
2. **Patches the Sampler import bug** in vllm-neuron 0.4.1
3. **Patches NxDI deepstack_vision_embeds** forward mismatch (required for Qwen3-VL)
4. **Installs the FastAPI proxy** (entrypoint.py) and dependencies

Unlike older approaches, there is **no DLAMI artifact extraction** and **no model-specific code baked in**.

In [ ]:
dockerfile = r"""# General-Purpose vLLM + Neuron SageMaker Container
# ==================================================
# Deploys any vllm-neuron supported model on SageMaker Neuron instances.
#
# Uses the pre-built vLLM Neuron DLC as base (has vLLM + vllm_neuron +
# NxDI pre-installed at tested versions). Adds SageMaker proxy on top.
#
# Supported models: Llama 2/3/4, Qwen 2.5/3/3-VL, Pixtral, and any model
# listed in vllm-neuron's NEURON_SUPPORTED_MODELS.
#
# Supported instances: ml.inf2.*, ml.trn1.* (SDK 2.28), ml.trn2.* (SDK 2.29)
#
# Configuration is passed entirely via environment variables:
#   SM_MODEL_ID        - HuggingFace model ID or local path (required)
#   SM_TP_DEGREE       - Tensor parallel degree (default: auto-detect)
#   SM_MAX_MODEL_LEN   - Max sequence length (default: 4096)
#   SM_MAX_NUM_SEQS    - Max concurrent sequences (default: 4)
#   SM_NEURON_CONFIG    - JSON override_neuron_config (optional)
#   SM_VLLM_ARGS       - Additional vllm serve arguments (optional)
#
# Build:
#   docker build --build-arg REGION=us-west-2 -t neuron-vllm-sagemaker .
#
# Version matrix:
#   SDK 2.28: vLLM 0.13.0, NxDI 0.8.x, vllm-neuron 0.4.1 (inf2/trn1/trn2)
#   SDK 2.29: vLLM 0.16.0, NxDI 0.9.x, vllm-neuron 0.5.0 (trn2 only)

# ============================================================
# Build arguments
# ============================================================
ARG REGION=us-east-1

# Pre-built vLLM Neuron DLC (has vLLM + vllm_neuron + NxDI pre-installed)
ARG BASE_IMAGE=public.ecr.aws/neuron/pytorch-inference-vllm-neuronx:0.13.0-neuronx-py312-sdk2.28.0-ubuntu24.04

FROM ${BASE_IMAGE}

# ============================================================
# Environment defaults (can be overridden at deploy time)
# ============================================================
ENV VLLM_NEURON_FRAMEWORK=neuronx-distributed-inference
ENV NEURON_COMPILE_CACHE_URL=/var/tmp/neuron-compile-cache
ENV SAGEMAKER_BIND_TO_PORT=8080
# Disable on-device sampling to avoid NKI cumsum kernels that generate
# ExtISA instructions incompatible with inf2 (PSUM constraint violation).
# Safe for all instances; sampling falls back to CPU-side top-k/top-p.
ENV NEURON_ON_DEVICE_SAMPLING_DISABLED=1

# SageMaker requires this label for BYOC containers
LABEL com.amazonaws.sagemaker.capabilities.accept-bind-to-port=true

# ============================================================
# Patch vllm_neuron Sampler import bug
# vllm_neuron 0.4.1 imports the sampler MODULE instead of the
# Sampler CLASS, causing TypeError when on-device sampling is
# disabled. Fix: import the class directly.
# ============================================================
RUN sed -i 's|from vllm.v1.sample import sampler as Sampler|from vllm.v1.sample.sampler import Sampler|' \
    /opt/vllm/vllm_neuron/worker/neuronx_distributed_model_loader.py && \
    echo "Patched Sampler import in vllm_neuron"

# ============================================================
# Patch NxDI deepstack_vision_embeds forward mismatch
#
# NxDI 0.8.x has two bugs that drop the 25th argument
# (deepstack_vision_embeds) needed by Qwen3-VL models:
#
# Bug 1: ImageToTextModelWrapper._forward_with_pad() uses
#   args[5:24] (24 total args) but Qwen3-VL compiles with
#   25 args. Fix: args[5:24] -> args[5:]
#
# Bug 2: ModelWrapper.pad_inputs() CTE vision re-padding uses
#   args[:22] + padded_vision, dropping args[24:].
#   Fix: preserve remaining args after the padded vision.
# ============================================================
RUN SITE_PKG=$(python3 -c "import site; print(site.getsitepackages()[0])") && \
    # Bug 1: _forward_with_pad args[5:24] -> args[5:]
    sed -i 's|for i, arg in enumerate(args\[5:24\]):|for i, arg in enumerate(args[5:]):|' \
        "$SITE_PKG/neuronx_distributed_inference/models/image_to_text_model_wrapper.py" && \
    # Bug 2: pad_inputs CTE vision re-padding - preserve deepstack args
    sed -i 's|args = (\*args\[:22\], \*padded_args)|args = (*args[:22], *padded_args, *args[24:])|' \
        "$SITE_PKG/neuronx_distributed_inference/models/model_wrapper.py" && \
    echo "Patched NxDI deepstack_vision_embeds forward mismatch"

# ============================================================
# Install SageMaker proxy dependencies
# (FastAPI + uvicorn for the /ping + /invocations bridge)
# ============================================================
RUN pip install --no-cache-dir qwen-vl-utils pillow httpx fastapi "uvicorn[standard]" && \
    rm -rf ~/.cache/pip/*

# ============================================================
# SageMaker entrypoint
#
# Bridges SageMaker's /ping + /invocations contract to
# vLLM's native /health + /v1/chat/completions endpoints.
# ============================================================
COPY entrypoint.py /opt/ml/code/entrypoint.py

# Create necessary directories
RUN mkdir -p /opt/ml/model /var/tmp/neuron-compile-cache

EXPOSE 8080

ENTRYPOINT ["python3", "/opt/ml/code/entrypoint.py"]
CMD ["serve"]
"""

with open(f"{BUILD_DIR}/Dockerfile", "w") as f:
    f.write(dockerfile)

print(f"Dockerfile written to {BUILD_DIR}/Dockerfile")
print(f"Size: {len(dockerfile)} bytes")

## Step 4: Create the Entrypoint (entrypoint.py)

The entrypoint does three things:

1. **Applies on-disk NxDI patches** before launching vLLM (tie_word_embeddings, vision QKV split)
2. **Creates a writable model overlay** (SageMaker mounts `/opt/ml/model` read-only, but NxDI needs to write compiled artifacts)
3. **Runs a FastAPI proxy** on port 8080 that forwards SageMaker requests to vLLM's native OpenAI API

The file is downloaded from this repository's `sagemaker-byoc/entrypoint.py`.

In [ ]:
import urllib.request

ENTRYPOINT_URL = "https://raw.githubusercontent.com/jimburtoft/NeuronStuff/jimburtoft/qwen3-vl-4b/sagemaker-byoc/entrypoint.py"

print("Downloading entrypoint.py...")
urllib.request.urlretrieve(ENTRYPOINT_URL, f"{BUILD_DIR}/entrypoint.py")

# Verify
size = os.path.getsize(f"{BUILD_DIR}/entrypoint.py")
print(f"entrypoint.py downloaded: {size} bytes")
print(f"\nBuild directory contents:")
for f in os.listdir(BUILD_DIR):
    fpath = os.path.join(BUILD_DIR, f)
    print(f"  {f} ({os.path.getsize(fpath)} bytes)")

## Step 5: Build and Push the Container

This step:
1. Creates the ECR repository (if it doesn't exist)
2. Authenticates Docker to ECR
3. Builds the container (~5 minutes -- mostly pip install of FastAPI/httpx)
4. Pushes to ECR

**Disk space**: The build requires ~15 GB. The base vLLM DLC is ~10 GB.

In [ ]:
%%bash -s "$ACCOUNT_ID" "$REGION" "$ECR_REPO"
ACCOUNT_ID=$1
REGION=$2
ECR_REPO=$3

# Create ECR repo if needed
aws ecr describe-repositories --repository-names $ECR_REPO --region $REGION 2>/dev/null || \
  aws ecr create-repository --repository-name $ECR_REPO --region $REGION

echo "ECR repository ready: $ECR_REPO"

In [ ]:
%%bash -s "$ACCOUNT_ID" "$REGION" "$ECR_REPO" "$ECR_TAG" "$BUILD_DIR"
set -e
ACCOUNT_ID=$1
REGION=$2
ECR_REPO=$3
ECR_TAG=$4
BUILD_DIR=$5
IMAGE_URI="${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com/${ECR_REPO}:${ECR_TAG}"

echo "=== Authenticating to ECR ==="
# Public ECR (base image)
aws ecr-public get-login-password --region us-east-1 | \
  docker login --username AWS --password-stdin public.ecr.aws

# Your ECR registry
aws ecr get-login-password --region $REGION | \
  docker login --username AWS --password-stdin ${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com

echo ""
echo "=== Building container ==="
echo "This takes ~5 minutes (mostly pip install of FastAPI/httpx)."
echo ""

cd $BUILD_DIR
docker build -t $IMAGE_URI .

echo ""
echo "=== Pushing to ECR ==="
docker push $IMAGE_URI

echo ""
echo "Container pushed: $IMAGE_URI"

## Step 6: Verify the Container

In [ ]:
%%bash -s "$IMAGE_URI"
IMAGE_URI=$1

echo "=== Verifying container imports ==="
docker run --rm --entrypoint python3 $IMAGE_URI -c "
import torch_neuronx; print(f'torch-neuronx {torch_neuronx.__version__}')
import vllm; print(f'vLLM {vllm.__version__}')
import vllm_neuron; print('vllm_neuron plugin loaded')
import neuronx_distributed_inference; print('NxD Inference loaded')
import fastapi; print(f'FastAPI {fastapi.__version__}')
print('All imports OK')
"

## Step 7: Deploy to SageMaker

Now use the endpoint notebook (`qwen3vl_4b_sagemaker_endpoint.ipynb`) to deploy this container.

Key settings for the endpoint notebook:

```python
ECR_REPO = "neuron-vllm-sagemaker"
ECR_TAG = "sdk2.28-v8"
INSTANCE_TYPE = "ml.inf2.8xlarge"
INFERENCE_AMI_VERSION = "al2-ami-sagemaker-inference-neuron-2"  # CRITICAL
```

The `InferenceAmiVersion` parameter is **required** -- it provides Neuron host driver v2.19, which supports the Extended ISA instructions generated by the NxD Inference compiler.

In [ ]:
print("=" * 60)
print("BUILD COMPLETE")
print("=" * 60)
print(f"")
print(f"Container: {IMAGE_URI}")
print(f"")
print(f"Next steps:")
print(f"  1. Upload model weights to S3")
print(f"  2. Open qwen3vl_4b_sagemaker_endpoint.ipynb")
print(f"  3. Set ECR_REPO = \"{ECR_REPO}\" and ECR_TAG = \"{ECR_TAG}\"")
print(f"  4. Run the deployment cells")
print(f"")
print(f"IMPORTANT: The endpoint config MUST include:")
print(f"  InferenceAmiVersion = 'al2-ami-sagemaker-inference-neuron-2'")

## Technical Notes

### Package Versions (SDK 2.28 DLC)

| Package | Version | Source |
|---------|---------|--------|
| PyTorch | 2.9.0 | DLC base image |
| torch-neuronx | 2.9.0.x | DLC base image |
| neuronx-cc | 2.22.x | DLC base image |
| neuronx-distributed-inference | 0.8.x | DLC base image |
| vLLM | 0.13.0 | DLC base image |
| vllm-neuron | 0.4.1 | DLC base image |

### Supported Models (vllm-neuron 0.4.1)

- Llama 2/3.1/3.3
- Qwen 2.5, Qwen 3
- Qwen2-VL, **Qwen3-VL**
- Pixtral

### Adapting for SDK 2.29 (trn2 only)

For SDK 2.29, change the base image:

```
ARG BASE_IMAGE=public.ecr.aws/neuron/pytorch-inference-vllm-neuronx:0.16.0-neuronx-py312-sdk2.29.0-ubuntu24.04
```

The Sampler import patch is not needed (fixed in vllm-neuron 0.5.0). The deepstack patches may still be needed depending on the NxDI version.

### Known Issues

1. **Sampler import bug** (SDK 2.28 only): vllm-neuron 0.4.1 imports the sampler module instead of the class. Fixed by sed patch.
2. **deepstack_vision_embeds** (NxDI 0.8.x): Forward methods truncate the 25th argument. Fixed by two sed patches.
3. **tie_word_embeddings** (Qwen3-VL): Missing method on VLM class. Fixed at startup by entrypoint.py.
4. **Vision QKV split** (NxDI 0.8.x + Qwen3-VL): Fused QKV mapped but `fused_qkv=False` default. Fixed at startup by entrypoint.py.
5. **Read-only /opt/ml/model**: SageMaker mounts model data read-only. entrypoint.py creates writable overlay at `/tmp/model`.

### References

- [vllm-neuron GitHub](https://github.com/vllm-project/vllm-neuron) -- official plugin
- [NxD Inference docs](https://awsdocs-neuron.readthedocs-hosted.com/en/latest/libraries/nxd-inference/index.html) -- model configuration
- [SageMaker InferenceAmiVersion](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_ProductionVariant.html) -- host AMI selection